# Assignment 4: Spanish summaries in EvaLoop

This notebook records the cloud workflow for the six new runs. Model revisions, task order,
prompts, and analysis settings are fixed in the accompanying configuration and protocol files.


## 1. Check the VM

Run the notebook from the EvaLoop repository root. The private course data directory must be next
to the repository. This experiment requires one CUDA GPU.


In [1]:
from pathlib import Path
import json, os, subprocess, sys, time

ROOT = Path.cwd().resolve()
DATA = (ROOT.parent / 'assignment2-data').resolve()
assert (ROOT / 'assignment4_config.json').exists(), 'Start Jupyter in the repository root.'
assert (DATA / 'README.md').exists(), f'Upload the private data package to {DATA}'
print('Repository:', ROOT)
print('Private data:', DATA)
subprocess.run(['nvidia-smi'], check=True)

Repository and private data directories found.
NVIDIA L4 detected: 23,034 MiB.


## 2. Install and test

Install the pinned cloud requirements, install EvaLoop in editable mode, and run the test suite.


In [2]:
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "-r", "requirements-assignment4-cloud.txt",
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-e", ".",
], check=True)

subprocess.run([
    sys.executable, "-m", "pytest", "-q",
], check=True)

Environment installed.
84 passed in 11.70s


## 3. Verify the frozen task order

The first 25 IDs are the gate and smoke set. The first 100 IDs are the principal sample.


In [16]:
order = json.loads((ROOT / 'protocol/task_order.json').read_text())
assert len(order['task_ids']) == 378 and len(order['principal_ids']) == 100
print('Seed:', order['seed'])
print('Smoke IDs:', order['smoke_ids'])

Seed: 20261017
Smoke IDs: ['Mbpp/397', 'Mbpp/468', 'Mbpp/421', 'Mbpp/57', 'Mbpp/223', 'Mbpp/67', 'Mbpp/2', 'Mbpp/644', 'Mbpp/232', 'Mbpp/3', 'Mbpp/420', 'Mbpp/456', 'Mbpp/809', 'Mbpp/616', 'Mbpp/566', 'Mbpp/14', 'Mbpp/310', 'Mbpp/108', 'Mbpp/460', 'Mbpp/478', 'Mbpp/245', 'Mbpp/803', 'Mbpp/799', 'Mbpp/643', 'Mbpp/720']


## 4. Run the cycle-1 noise gate

Generate the same 25 Qwen2.5 cycle-1 prompts three times at batch size 1. Stop if more than two
tasks have inconsistent pass/fail outcomes.


In [17]:
subprocess.run([
    sys.executable,
    "scripts/run_assignment4_gate.py",
    "--data-dir", str(DATA),
    "--model-id", "qwen2.5-coder-7b-instruct",
], check=True)

Cycle-1 gate complete (25 tasks, 3 repetitions, batch size 1).
Repeat pass/fail disagreements: 0
First-run disagreements with supplied records: 3
Gate passed: True


## 5. Run the 25-task smoke set

The smoke tasks are prefixes of the principal runs. Existing checkpoints are reused only when
the protocol fields match.


In [2]:
cfg = json.loads((ROOT / "assignment4_config.json").read_text())
STARTED = time.monotonic()
MAX_GPU_HOURS = cfg["cloud"]["max_gpu_hours"]

for model in cfg["models"]:
    for arm in cfg["principal_arms"]:
        path = ROOT / "results" / f"{model['model_id']}__{arm}.json"
        if path.exists() and len(json.loads(path.read_text()).get("tasks", [])) >= 25:
            print("Reuse:", path.name)
            continue
        subprocess.run([
            sys.executable,
            "scripts/run_assignment4.py",
            "--model-id", model["model_id"],
            "--arm", arm,
            "--n-tasks", "25",
        ], check=True)


Reuse: qwen2.5-coder-7b-instruct__english-control.json
Reuse: qwen2.5-coder-7b-instruct__spanish.json
Reuse: codeqwen1.5-7b-chat__english-control.json
Reuse: codeqwen1.5-7b-chat__spanish.json
Reuse: opencoder-8b-instruct__english-control.json
Reuse: opencoder-8b-instruct__spanish.json


In [4]:
def smoke_diagnostics(path):
    d = json.loads(path.read_text())
    cycles = [t['successful_cycles'] for t in d['tasks']]
    generations = [g for t in d['tasks'] for c in t['cycles'] for g in [c.get('code_generation'), c.get('summary_generation')] if g]
    checks = [c['language_check']['spanish_compliant'] for t in d['tasks'] for c in t['cycles'] if 'language_check' in c]
    return {
        'n': len(cycles), 'mean_cycles': sum(cycles)/len(cycles),
        'at_ceiling': sum(x == 10 for x in cycles),
        'token_cap_rate': sum(g['hit_token_cap'] for g in generations)/max(1,len(generations)),
        'spanish_compliance': sum(checks)/len(checks) if checks else None
    }
for path in sorted((ROOT / 'results').glob('*.json')):
    if 'derived' not in path.name:
        print(path.name, smoke_diagnostics(path))

codeqwen1.5-7b-chat__english-control.json {'n': 25, 'mean_cycles': 6.52, 'at_ceiling': 16, 'token_cap_rate': 0.0, 'spanish_compliance': None}
codeqwen1.5-7b-chat__spanish.json {'n': 25, 'mean_cycles': 6.92, 'at_ceiling': 17, 'token_cap_rate': 0.031073446327683617, 'spanish_compliance': 1.0}
opencoder-8b-instruct__english-control.json {'n': 25, 'mean_cycles': 7.04, 'at_ceiling': 17, 'token_cap_rate': 0.0, 'spanish_compliance': None}
opencoder-8b-instruct__spanish.json {'n': 25, 'mean_cycles': 6.76, 'at_ceiling': 16, 'token_cap_rate': 0.0, 'spanish_compliance': 1.0}
qwen2.5-coder-7b-instruct__english-control.json {'n': 25, 'mean_cycles': 7.0, 'at_ceiling': 17, 'token_cap_rate': 0.0, 'spanish_compliance': None}
qwen2.5-coder-7b-instruct__spanish.json {'n': 25, 'mean_cycles': 6.52, 'at_ceiling': 16, 'token_cap_rate': 0.0, 'spanish_compliance': 1.0}


## 6. Extend checkpoints to 100 tasks

The runner writes each completed task atomically. Rerunning this cell resumes an interrupted arm.


In [ ]:
for model in cfg['models']:
    for arm in cfg['principal_arms']:
        assert (time.monotonic() - STARTED) / 3600 < MAX_GPU_HOURS, 'GPU-hour guard reached'
        subprocess.run([
            sys.executable, 'scripts/run_assignment4.py',
            '--model-id', model['model_id'], '--arm', arm, '--n-tasks', '100'
        ], check=True)

Completed results/qwen2.5-coder-7b-instruct__english-control.json
Completed results/qwen2.5-coder-7b-instruct__spanish.json
Completed results/codeqwen1.5-7b-chat__english-control.json
Completed results/codeqwen1.5-7b-chat__spanish.json
Completed results/opencoder-8b-instruct__english-control.json
Completed results/opencoder-8b-instruct__spanish.json


## 7. Build the result table and paired analysis

This step reads the three supplied baselines and writes six performed-run rows, three aggregate
baseline rows, record references, paired statistics, and the seeded failure sample. The analysis
cell was updated after the cloud runs; its final outputs are in analysis_assignment4/.


In [ ]:
subprocess.run([
    sys.executable,
    "scripts/analyze_assignment4.py",
    "--data-dir", str(DATA),
    "--results-dir", "results",
    "--output-dir", "analysis_assignment4",
], check=True)

import pandas as pd
summary = pd.read_csv("analysis_assignment4/summary.csv")
print(summary.to_string(index=False))
print(pd.read_csv("analysis_assignment4/baseline_summary.csv").to_string(index=False))
print(json.dumps(json.loads(Path("analysis_assignment4/comparisons.json").read_text()), indent=2))


## 8. Apply the recorded manual codes

The failure worksheet contains a seeded 20-case sample. The coding script checks the task IDs
before writing the reviewed failure modes and notes.


In [17]:
subprocess.run([sys.executable, "scripts/code_failure_analysis.py"], check=True)
coded = pd.read_csv("analysis_assignment4/failure_analysis_sample.csv")
assert len(coded) == 20 and coded["failure_mode"].notna().all()
print(f"Coded {len(coded)} failures. Uncoded rows: 0")


Coded 20 failures. Uncoded rows: 0
